In [1]:
# pip install pillow

<h1 style="text-align:left; letter-spacing:2px; font-weight:700;">
Environment Setup
</h1>

In [7]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import subprocess
import shutil
import glob

<h1 style="text-align:left; letter-spacing:2px; font-weight:700;">
Generate Example Animation Frames
</h1>

In [17]:
os.makedirs("Example_Images", exist_ok=True)

x = np.linspace(-5, 5, 40)
y = np.linspace(-5, 5, 40)
X, Y = np.meshgrid(x, y)
R = np.sqrt(X**2 + Y**2)

fig = plt.figure(figsize=(4, 3), dpi=50)
ax = fig.add_subplot(111, projection='3d')

for i in range(60):
    ax.clear()
    
    phase = 2 * np.pi * (i / 60.0)
    Z = np.sin(R - phase) * np.exp(-0.1 * R)
    
    ax.plot_surface(X, Y, Z, cmap='plasma', edgecolor='none')
    
    ax.set_zlim(-1, 1)
    ax.set_axis_off()
    
    ax.view_init(elev=30, azim=i * 6)
    
    filename = os.path.join("Example_Images", f"frame_{i:02d}.png")
    plt.savefig(filename, dpi=50, transparent=True, bbox_inches='tight')

plt.close(fig)

<h1 style="text-align:left; letter-spacing:2px; font-weight:700;">
Render Transparent GIF
</h1>

In [18]:
image_folder = "Example_Images"
png_files = sorted([os.path.join(image_folder, f) for f in os.listdir(image_folder) if f.endswith('.png')])
frames = []
for png_file in png_files:
    img = Image.open(png_file).convert("RGBA")
    alpha = img.getchannel("A")
    img_rgb = Image.new("RGB", img.size, (255, 255, 255))
    img_rgb.paste(img, mask=alpha)
    imgP = img_rgb.convert("P", palette=Image.ADAPTIVE, colors=255, dither=Image.NONE)
    mask = alpha.point(lambda p: 255 if p == 0 else 0)
    imgP.paste(255, mask)
    frames.append(imgP)
frames[0].save("Transparent Movie.gif", save_all=True, append_images=frames[1:], duration=60, loop=0, transparency=255, disposal=2, optimize=False)

<h1 style="text-align:left; letter-spacing:2px; font-weight:700;">
Render Transparent MOV
</h1>

In [ ]:
image_folder = "Example_Images"
temp_folder = os.path.join(image_folder, "_seq")
output_mov = "Transparent Movie.mov"

if os.path.isdir(temp_folder):
    shutil.rmtree(temp_folder)
os.makedirs(temp_folder, exist_ok=True)

files = sorted(glob.glob(os.path.join(image_folder, "frame_*.png")))

i = 0
for src in files:
    dst = os.path.join(temp_folder, f"frame_{i:04d}.png")
    shutil.copy2(src, dst)
    i += 1

cmd = [
    "ffmpeg",
    "-y",
    "-framerate", "12", # Control the frame rate of the output video, 12 per second (fps) in this case.
    "-i", os.path.join(temp_folder, "frame_%04d.png"),
    "-c:v", "prores_ks",
    "-profile:v", "4",
    "-pix_fmt", "yuva444p10le",
    output_mov
]
subprocess.run(cmd, check=True)

ffmpeg version 8.0.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.6.3.2)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/8.0.1_1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags= --enable-ffplay --enable-gpl --enable-libsvtav1 --enable-libopus --enable-libx264 --enable-libmp3lame --enable-libdav1d --enable-libvpx --enable-libx265 --enable-videotoolbox --enable-audiotoolbox --enable-neon
  libavutil      60.  8.100 / 60.  8.100
  libavcodec     62. 11.100 / 62. 11.100
  libavformat    62.  3.100 / 62.  3.100
  libavdevice    62.  1.100 / 62.  1.100
  libavfilter    11.  4.100 / 11.  4.100
  libswscale      9.  1.100 /  9.  1.100
  libswresample   6.  1.100 /  6.  1.100
Input #0, image2, from 'Example_Images/_seq/frame_%04d.png':
  Duration: 00:00:05.00, start: 0.000000, bitrate: N/A
  Stream #0:0: Video: png, rgba(pc, gbr/unknown/unknown), 125x125 [SAR 1969:1969 DAR 1:1], 12 fps, 12 tb

CompletedProcess(args=['ffmpeg', '-y', '-framerate', '12', '-i', 'Example_Images/_seq/frame_%04d.png', '-c:v', 'prores_ks', '-profile:v', '4', '-pix_fmt', 'yuva444p10le', 'Transparent Movie.mov'], returncode=0)